# Ettin: retain a pretrained small encoder and train its embeddings

    Self-contained A100 / VS Code Colab notebook. Run cells top to bottom. All implementation lives here;
    no repository checkout, CLI runner, external Python module, or second notebook is required.
    Checkpoints, datasets, target caches and reports persist under `MyDrive/encoder_experiments/RUN_NAME`.

    This is our experiment, not a reproduction of ModernBERT-small-v2. Defaults are a bounded English pilot,
    not enough data to establish broad or multilingual quality. The same pairs, supervisor and evaluations
    are used in both experiments. Full-model training loss alone is never the acceptance criterion.

In [2]:
# Install into this notebook's kernel. Keep Colab's CUDA-enabled PyTorch.
# Restart the kernel after this cell if it already imported different versions.
%pip install -q transformers==5.1.0 datasets==4.5.0 accelerate==1.12.0 sentence-transformers==5.2.2 safetensors

In [3]:
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_HOME'] = '/content/hf-cache' if os.path.isdir('/content') else '/tmp/encoder-hf-cache'
import copy, gc, hashlib, json, math, random, shutil, time, uuid
from pathlib import Path
from collections import defaultdict
from contextlib import nullcontext
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint
from transformers import AutoConfig, AutoModel, AutoTokenizer, get_cosine_schedule_with_warmup
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file, save_file
from tqdm.auto import tqdm
from google.colab import drive

drive.mount('/content/drive')
DEVICE = 'cuda'
assert torch.cuda.is_available(), 'Select an A100 GPU runtime in Colab.'
assert torch.cuda.is_bf16_supported(), 'This notebook uses BF16 on a supported GPU.'
print(torch.cuda.get_device_name(), 'PyTorch:', torch.__version__)
torch.set_float32_matmul_precision('high')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
NVIDIA A100-SXM4-40GB PyTorch: 2.11.0+cu128


## Configuration and resume
    Choose a unique run name per configuration. Default sequence length is 256 for short-text experiments.
    Start with 2,000 rows / 20 steps to debug if needed, then use a new name for the pilot.
    BF16 autocast uses FP32 trainable weights and optimizer moments. An A100 is expected.
    Stop normally to save the last completed optimizer step; after a hard disconnect, resume from the last
    completed Drive checkpoint. Resume includes optimizer, schedule, sampler and random states.

In [4]:
# Edit this cell. Use a NEW RUN_NAME for changed architecture/data/loss settings.
# Stop a training cell, edit a function, then rerun that definition and training cell.
RUN_NAME = 'ettin-pilot-01'
ROOT = Path('/content/drive/MyDrive/encoder_experiments') / RUN_NAME
ROOT.mkdir(parents=True, exist_ok=True)
LOCAL = Path('/content/encoder-work') / RUN_NAME
LOCAL.mkdir(parents=True, exist_ok=True)
CFG = dict(
    experiment='ettin', seed=42, max_length=256,
    train_rows=20000, validation_rows=256, test_rows=256,
    batch_size=32, teacher_batch_size=32, accumulation=1,
    teacher_steps=600, student_steps=1000, matryoshka_steps=300,
    teacher_lr=2e-5, student_lr=3e-5, weight_decay=0.01,
    temperature=0.05, contrastive_weight=1.0, alignment_weight=1.0,
    relational_weight=0.2, warmup_fraction=0.1,
    checkpoint_every=100, eval_every=100, log_every=10,
    validation_batches=4, gradient_checkpointing=True,
    student_layers=8, ettin_size='32m', run_matryoshka=False,
    # This is the ACTUAL contrastive batch: accumulation does not enlarge negatives.
)
GTE = dict(id='Alibaba-NLP/gte-modernbert-base', revision='e7f32e3c00f91d699e8c43b53106206bcc72bb22')
NEO = dict(id='chandar-lab/NeoBERT', revision='5424c8efeea6491b151d62dee55a752165407430')
ETTIN = {
    '32m': dict(id='jhu-clsp/ettin-encoder-32m', revision='1b8ba06455dd44f80fc9c1ca9e22806157a57379'),
    '68m': dict(id='jhu-clsp/ettin-encoder-68m', revision='ac19ae4bc51093b31c475665ac872a936d056cc2'),
}[CFG['ettin_size']]
DATA = dict(id='sentence-transformers/msmarco-msmarco-MiniLM-L6-v3',
            revision='0d54352548089199bde15ad7e06efe895dc80b56', subset='triplet')
NANO = dict(id='sentence-transformers/NanoBEIR-en', revision='beb106fbcfaa599c508c667041bf8c85fd78736b')

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_everything(CFG['seed'])
def digest(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True, ensure_ascii=False).encode()).hexdigest()
def text_id(s):
    return hashlib.sha256(' '.join(s.split()).casefold().encode()).hexdigest()
def write_json(path, obj):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp-' + uuid.uuid4().hex)
    tmp.write_text(json.dumps(obj, indent=2, ensure_ascii=False))
    os.replace(tmp, path)
contract = dict(config=CFG, gte=GTE, neo=NEO, ettin=ETTIN, data=DATA, nano=NANO, implementation=1)
CONTRACT_ID = digest(contract)
if (ROOT/'config.json').exists():
    assert json.loads((ROOT/'config.json').read_text()) == contract, 'Config changed: choose a new RUN_NAME.'
else:
    write_json(ROOT/'config.json', contract)
print(ROOT, '\nConfig ID:', CONTRACT_ID)

/content/drive/MyDrive/encoder_experiments/ettin-pilot-01 
Config ID: bda866f7e45d3ef6e24f79f5c12e25d48dd64ec157f8845d5bff9963b5f95dc3


## Shared benchmark and training data

In [5]:
# A pinned, labeled retrieval benchmark. Never used for optimization or early stopping.
BENCHMARKS = {}
def load_benchmarks():
    if BENCHMARKS:
        return BENCHMARKS
    for name in ('NanoMSMARCO', 'NanoHotpotQA'):
        parts = {part: load_dataset(NANO['id'], part, split=name, revision=NANO['revision'])
                 for part in ('queries', 'corpus', 'qrels')}
        queries = {str(r['_id']): r['text'] for r in parts['queries'] if r['text'].strip()}
        corpus = {str(r['_id']): r['text'] for r in parts['corpus'] if r['text'].strip()}
        relevant = defaultdict(set)
        for r in parts['qrels']:
            ids = r['corpus-id'] if isinstance(r['corpus-id'], list) else [r['corpus-id']]
            relevant[str(r['query-id'])].update(str(x) for x in ids)
        BENCHMARKS[name] = dict(queries=queries, corpus=corpus, relevant=dict(relevant))
    return BENCHMARKS

@torch.inference_mode()
def encode_texts(model, texts, batch_size=None, dim=None):
    model.eval()
    result = []
    for start in range(0, len(texts), batch_size or CFG['teacher_batch_size']):
        with torch.autocast('cuda', dtype=torch.bfloat16):
            z = model(texts[start:start+(batch_size or CFG['teacher_batch_size'])])
        z = z.float().cpu()
        assert torch.isfinite(z).all(), 'Non-finite embeddings'
        result.append(F.normalize(z[:, :dim], dim=-1) if dim else F.normalize(z, dim=-1))
    return torch.cat(result)

def retrieval_metrics(q, p, relevant, query_ids, corpus_ids):
    scores = q @ p.T
    ranking = scores.argsort(dim=1, descending=True)
    sums = defaultdict(float)
    for i, qid in enumerate(query_ids):
        truth = relevant[qid]
        top = [corpus_ids[j] for j in ranking[i, :10].tolist()]
        hits = [int(x in truth) for x in top]
        sums['hit@1'] += hits[0]
        sums['hit@5'] += bool(sum(hits[:5]))
        sums['recall@10'] += sum(hits) / len(truth)
        sums['mrr@10'] += next((1/(j+1) for j,h in enumerate(hits) if h), 0)
        dcg = sum(h/math.log2(j+2) for j,h in enumerate(hits))
        ideal = sum(1/math.log2(j+2) for j in range(min(len(truth), 10)))
        sums['ndcg@10'] += dcg / ideal
    return {k: v/len(query_ids) for k,v in sums.items()}

def benchmark(model, label, dims=None):
    results = {}
    for name, b in load_benchmarks().items():
        corpus_ids = list(b['corpus'])
        query_ids = [qid for qid in b['queries'] if b['relevant'].get(qid)]
        assert all(b['relevant'][qid] <= set(corpus_ids) for qid in query_ids)
        q = encode_texts(model, [b['queries'][k] for k in query_ids])
        p = encode_texts(model, list(b['corpus'].values()))
        for dim in dims or [q.shape[1]]:
            metrics = retrieval_metrics(F.normalize(q[:,:dim], dim=-1), F.normalize(p[:,:dim], dim=-1),
                                        b['relevant'], query_ids, corpus_ids)
            results[f'{name}/{dim}'] = metrics
            print(label, name, dim, {k: round(v,4) for k,v in metrics.items()}, flush=True)
    write_json(ROOT/'evaluations'/f'{label}.json', dict(label=label, contract=CONTRACT_ID,
               benchmark=NANO, max_length=CFG['max_length'], results=results))
    return results

In [6]:
# Query-disjoint splits, followed by exact text-overlap filtering across splits.
# Exclude benchmark queries AND documents. This is exact deduplication, not semantic deduplication.
corpus_file = ROOT/'pairs.json'
if corpus_file.exists():
    corpus = json.loads(corpus_file.read_text())
else:
    excluded = {text_id(t) for b in load_benchmarks().values()
                for collection in ('queries','corpus') for t in b[collection].values()}
    caps = {'train': CFG['train_rows'], 'validation': CFG['validation_rows'], 'test': CFG['test_rows']}
    corpus = {s: [] for s in caps}
    seen_pairs = set()
    seen_queries = set()
    stream = load_dataset(DATA['id'], DATA['subset'], split='train', revision=DATA['revision'], streaming=True)
    stream = stream.shuffle(seed=CFG['seed'], buffer_size=10000)
    for r in tqdm(stream, desc='Select disjoint pairs', total=None):
        row = {k: r[k].strip() for k in ('query','positive','negative')}
        if not all(row.values()): continue
        hashes = {text_id(x) for x in row.values()}
        if hashes & excluded or len(hashes) != 3: continue
        key = digest(row)
        qid = text_id(row['query'])
        if key in seen_pairs or qid in seen_queries: continue
        seen_pairs.add(key); seen_queries.add(qid)
        bucket = int(qid[:8], 16) % 100
        split = 'validation' if bucket < 5 else 'test' if bucket < 10 else 'train'
        if len(corpus[split]) < caps[split]: corpus[split].append(row)
        if all(len(corpus[s]) >= caps[s] for s in caps): break
    assert all(len(corpus[s]) >= caps[s] for s in caps), 'Source exhausted; lower row caps.'
    # Protect test first, then validation. Shared documents cannot leak into training.
    held = {text_id(x) for r in corpus['test'] for x in r.values()}
    corpus['validation'] = [r for r in corpus['validation'] if not {text_id(x) for x in r.values()} & held]
    held |= {text_id(x) for r in corpus['validation'] for x in r.values()}
    corpus['train'] = [r for r in corpus['train'] if not {text_id(x) for x in r.values()} & held]
    assert len(corpus['validation']) >= 32 and len(corpus['train']) >= 128
    write_json(corpus_file, corpus)
DATA_ID = digest(corpus)
print('Exact overlap filtering can reduce the requested sizes:', {k:len(v) for k,v in corpus.items()})
# IDs, rather than row position assumptions, connect text to teacher targets.
texts = list(dict.fromkeys(t for rows in corpus.values() for r in rows for t in r.values()))
text_to_index = {t:i for i,t in enumerate(texts)}

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Select disjoint pairs: 0it [00:00, ?it/s]

Exact overlap filtering can reduce the requested sizes: {'train': 19981, 'validation': 256, 'test': 256}


## GTE supervisor and teacher targets
    GTE-ModernBERT-base is Apache-2.0 and already trained for embeddings. It uses CLS pooling.
    Our students use masked mean pooling. Targets are normalized full 768-dimensional vectors;
    no arbitrary truncation or PCA of teacher targets is performed.

In [7]:
class GTEEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.tokenizer = AutoTokenizer.from_pretrained(GTE['id'], revision=GTE['revision'])
        self.backbone = AutoModel.from_pretrained(GTE['id'], revision=GTE['revision'],
                                                  attn_implementation='sdpa', dtype=torch.bfloat16)
        self.output_dim = self.backbone.config.hidden_size
    def forward(self, texts):
        tokens = self.tokenizer(texts, padding=True, truncation=True, max_length=CFG['max_length'],
                                return_tensors='pt').to(next(self.parameters()).device)
        # Published GTE-ModernBERT uses CLS pooling, not mean pooling.
        return self.backbone(**tokens).last_hidden_state[:,0].float()

supervisor = GTEEncoder().to(DEVICE).eval().requires_grad_(False)
print('Supervisor parameters:', sum(p.numel() for p in supervisor.parameters()))

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

Supervisor parameters: 149014272


In [8]:
gte_baseline = benchmark(supervisor, 'gte-baseline')

gte-baseline NanoMSMARCO 768 {'hit@1': 0.46, 'hit@5': 0.78, 'recall@10': 0.84, 'mrr@10': 0.5959, 'ndcg@10': 0.6552}
gte-baseline NanoHotpotQA 768 {'hit@1': 0.86, 'hit@5': 0.94, 'recall@10': 0.8, 'mrr@10': 0.8873, 'ndcg@10': 0.7768}


In [9]:
# Resume teacher extraction shard-by-shard. Validate each completed shard against its text IDs.
CACHE_CONTRACT = dict(data=DATA_ID, teacher=GTE, length=CFG['max_length'], pooling='cls',
                      normalized=True, storage='float32', format_version=1)
CACHE_ID = digest(CACHE_CONTRACT)
cache_dir = ROOT/'gte_cache'/CACHE_ID
cache_dir.mkdir(parents=True, exist_ok=True)
write_json(cache_dir/'contract.json', CACHE_CONTRACT)
SHARD_SIZE = 512
arrays = []
for start in tqdm(range(0, len(texts), SHARD_SIZE), desc='GTE target cache'):
    batch_texts = texts[start:start+SHARD_SIZE]
    path = cache_dir/f'{start:09d}.npz'
    expected_ids = np.asarray([hashlib.sha256(t.encode()).hexdigest() for t in batch_texts])
    if path.exists():
        with np.load(path, allow_pickle=False) as z:
            assert np.array_equal(z['ids'], expected_ids), 'Target/text identity mismatch'
            values = z['vectors'].copy()
    else:
        values = encode_texts(supervisor, batch_texts).numpy()
        local = LOCAL/(path.name+'.tmp.npz')
        np.savez(local, ids=expected_ids, vectors=values)
        tmp = path.with_suffix('.uploading')
        shutil.copyfile(local, tmp); os.replace(tmp, path)
    assert values.shape == (len(batch_texts), 768) and np.isfinite(values).all()
    assert np.allclose(np.linalg.norm(values,axis=1),1,atol=1e-4)
    arrays.append(values)
teacher_targets = torch.from_numpy(np.concatenate(arrays))
del arrays
write_json(cache_dir/'complete.json', dict(count=len(texts), cache_id=CACHE_ID))
print('Cached', teacher_targets.shape, 'on CPU; supervisor can now be released.')

GTE target cache:   0%|          | 0/120 [00:00<?, ?it/s]

Cached torch.Size([61134, 768]) on CPU; supervisor can now be released.


In [10]:
del supervisor
gc.collect(); torch.cuda.empty_cache()

## Editable model implementation

In [11]:
def mean_pool(hidden, mask):
    mask = mask.unsqueeze(-1).to(hidden.dtype)
    return (hidden.float()*mask).sum(1) / mask.sum(1).clamp_min(1)

# Input adapters belong to the model, so evaluation, training and export use the same tokenizer/pooling.
class EmbeddingModel(nn.Module):
    def __init__(self, backbone, tokenizer, output_dim, family, spec):
        super().__init__()
        self.backbone, self.tokenizer = backbone, tokenizer
        self.output_dim, self.family, self.spec = output_dim, family, copy.deepcopy(spec)
        self.max_length = CFG['max_length']
        # Ettin's alignment head is training-only. Retrieval always uses its native 384/512 dimensions.
        self.align = nn.Identity() if output_dim == 768 else nn.Linear(output_dim,768,bias=False)
    def forward(self, texts):
        tokens = self.tokenizer(texts, padding=True, truncation=True, max_length=self.max_length,
                                return_tensors='pt').to(next(self.parameters()).device)
        if self.family == 'neobert':
            h = self.backbone(tokens['input_ids'], tokens['attention_mask'])
        else:
            h = self.backbone(**tokens).last_hidden_state
        return mean_pool(h, tokens['attention_mask'])

def count_parameters(model):
    return dict(total=sum(p.numel() for p in model.parameters()),
                backbone=sum(p.numel() for p in model.backbone.parameters()),
                training_only_alignment=sum(p.numel() for p in model.align.parameters()))

In [12]:
def fresh_ettin():
    tok = AutoTokenizer.from_pretrained(ETTIN['id'],revision=ETTIN['revision'])
    backbone = AutoModel.from_pretrained(ETTIN['id'],revision=ETTIN['revision'],attn_implementation='sdpa')
    assert backbone.config.model_type == 'modernbert'
    if CFG['gradient_checkpointing']:
        backbone.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant':False})
    return EmbeddingModel(backbone,tok,backbone.config.hidden_size,'ettin',backbone.config.to_dict())

## Checkpointing and optimization
    Checkpoints are verified before use, and incomplete uploads are ignored. The latest two generations
    and validation-best generation are retained. Several GB of free Drive space may be needed.
    Resume settings are checked against the saved experiment. Intentional function edits remain possible;
    use a new run name when changing the experiment rather than repairing a bug.

    The loss combines true in-batch contrastive ranking (including each explicit hard negative), cosine
    alignment to a teacher, and pairwise similarity matching. Accumulation does not increase the negative pool.
    Validation selects the checkpoint; test pairs and NanoBEIR are reporting-only.

In [13]:
@torch.inference_mode()
def check_padding_invariance(model):
    model.eval()
    short = 'Reset my password.'
    long = 'A substantially longer sentence about weather, gardening, cooking and unrelated household tasks.'
    a = encode_texts(model, [short])
    b = encode_texts(model, [short, long])[:1]
    cosine = F.cosine_similarity(a,b).item()
    print('Single vs padded batch cosine:', cosine)
    assert cosine > 0.999, 'Padding/batching changed the embedding unexpectedly.'


In [14]:
# Completed generations are immutable. Upload to a temporary directory; COMMITTED.json is written last.
# Resume discovers the newest verified generation, rather than trusting a possibly stale "latest" pointer.
def file_sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(8*1024*1024),b''): h.update(chunk)
    return h.hexdigest()

def rng_state():
    return dict(python=random.getstate(), numpy=np.random.get_state(), cpu=torch.get_rng_state(),
                cuda=torch.cuda.get_rng_state_all())
def restore_rng(state):
    random.setstate(state['python']); np.random.set_state(state['numpy'])
    torch.set_rng_state(state['cpu'])
    if torch.cuda.is_available(): torch.cuda.set_rng_state_all(state['cuda'])

def save_checkpoint(stage, step, model, optimizer, scheduler, sampler, best_metric, best_step, total_steps):
    parent=ROOT/'checkpoints'/stage; parent.mkdir(parents=True,exist_ok=True)
    generation=f'{step:08d}-{uuid.uuid4().hex[:8]}'
    local=LOCAL/(generation+'.pt')
    torch.save(dict(stage=stage,step=step,total_steps=total_steps,contract=CONTRACT_ID,data_id=DATA_ID,
                    family=model.family,spec=model.spec,model=model.state_dict(),optimizer=optimizer.state_dict(),
                    scheduler=scheduler.state_dict(),sampler=sampler.state_dict(),rng=rng_state(),
                    best_metric=best_metric,best_step=best_step),local)
    tmp=parent/('.uploading-'+generation); tmp.mkdir()
    shutil.copyfile(local,tmp/'state.pt')
    sha=file_sha(local)
    assert file_sha(tmp/'state.pt') == sha, 'Drive checkpoint copy failed checksum verification'
    write_json(tmp/'COMMITTED.json',dict(step=step,sha256=sha,contract=CONTRACT_ID))
    os.replace(tmp,parent/generation)
    local.unlink()
    # Retain latest two plus the generation containing the best validation result.
    generations=sorted(p for p in parent.iterdir() if (p/'COMMITTED.json').exists() and not p.name.startswith('.'))
    keep={p.name for p in generations[-2:]}
    keep.update(p.name for p in generations if int(p.name.split('-')[0]) == best_step)
    for p in generations:
        if p.name not in keep: shutil.rmtree(p)
    return parent/generation

def latest_checkpoint(stage):
    parent=ROOT/'checkpoints'/stage
    if not parent.exists(): return None
    candidates=sorted((p for p in parent.iterdir() if not p.name.startswith('.') and (p/'COMMITTED.json').exists()),reverse=True)
    for path in candidates:
        try:
            manifest=json.loads((path/'COMMITTED.json').read_text())
            assert manifest['contract']==CONTRACT_ID
            assert file_sha(path/'state.pt')==manifest['sha256']
            return path
        except (OSError,ValueError,AssertionError) as e:
            print('Skipping invalid checkpoint',path,e)
    if candidates: raise RuntimeError('No valid checkpoint remains; inspect Drive files.')
    return None

def load_checkpoint(stage):
    path=latest_checkpoint(stage)
    if path is None: return None
    local=LOCAL/(stage+'-resume.pt'); shutil.copyfile(path/'state.pt',local)
    # Only our own checkpoints, never arbitrary downloaded pickle files.
    state=torch.load(local,map_location='cpu',weights_only=False)
    assert state['contract']==CONTRACT_ID and state['data_id']==DATA_ID
    return state

class StatefulSampler:
    def __init__(self,n,batch,seed):
        assert n>=batch
        self.n,self.batch=n,batch
        self.generator=torch.Generator().manual_seed(seed)
        self.order=torch.randperm(n,generator=self.generator); self.cursor=0; self.epoch=0
    def next(self):
        if self.cursor+self.batch>self.n:
            self.order=torch.randperm(self.n,generator=self.generator); self.cursor=0; self.epoch+=1
        ids=self.order[self.cursor:self.cursor+self.batch].tolist(); self.cursor+=self.batch
        return ids
    def state_dict(self):
        return dict(n=self.n,batch=self.batch,order=self.order,cursor=self.cursor,epoch=self.epoch,generator=self.generator.get_state())
    def load_state_dict(self,s):
        assert (self.n,self.batch)==(s['n'],s['batch'])
        self.order=s['order'];self.cursor=s['cursor'];self.epoch=s['epoch'];self.generator.set_state(s['generator'])

In [15]:
def flatten_pairs(rows):
    # This ordering is used for BOTH students and targets.
    return [r[k] for k in ('query','positive','negative') for r in rows]

def contrastive_loss(z, rows, dim):
    n=len(rows); z=F.normalize(z[:,:dim].float(),dim=-1)
    q,p,neg=z.split(n)
    docs=torch.cat((p,neg))
    logits=q@docs.T/CFG['temperature']
    doc_ids=[text_id(r[k]) for k in ('positive','negative') for r in rows]
    pos_ids=[text_id(r['positive']) for r in rows]
    query_ids=[text_id(r['query']) for r in rows]
    # Exclude duplicated positives and all alternative passages of duplicate queries from negatives.
    valid=torch.ones((n,2*n),device=z.device,dtype=torch.bool)
    for i in range(n):
        for j in range(2*n):
            if j!=i and (doc_ids[j]==pos_ids[i] or (j%n!=i and query_ids[j%n]==query_ids[i])):
                valid[i,j]=False
    logits=logits.masked_fill(~valid,torch.finfo(logits.dtype).min)
    return F.cross_entropy(logits,torch.arange(n,device=z.device))

def objective(model, raw, targets, rows, dimensions=None):
    z=F.normalize(raw.float(),dim=-1); target=F.normalize(targets.float(),dim=-1)
    aligned=F.normalize(model.align(raw).float(),dim=-1)
    align=(1-(aligned*target).sum(-1)).mean()
    pairmask=~torch.eye(len(z),device=z.device,dtype=torch.bool)
    relation=F.mse_loss((z@z.T)[pairmask],(target@target.T)[pairmask])
    dims=dimensions or [model.output_dim]
    contrastive=torch.stack([contrastive_loss(raw,rows,d) for d in dims]).mean()
    loss=CFG['contrastive_weight']*contrastive+CFG['alignment_weight']*align+CFG['relational_weight']*relation
    return loss,dict(contrastive=contrastive.detach().item(),alignment=align.detach().item(),
                     relational=relation.detach().item())

def targets_for(rows, teacher=None):
    batch_texts=flatten_pairs(rows)
    if teacher is None:
        return teacher_targets[[text_to_index[t] for t in batch_texts]].to(DEVICE)
    teacher.eval()
    with torch.no_grad(),torch.autocast('cuda',dtype=torch.bfloat16):
        return F.normalize(teacher(batch_texts).float(),dim=-1)

@torch.inference_mode()
def validate(model, teacher=None, split='validation'):
    # Fixed subset for comparable early stopping; complete held-out test is evaluated separately.
    model.eval(); rows=corpus[split]
    if split=='validation': rows=rows[:CFG['validation_batches']*CFG['batch_size']]
    q,p=[],[]; losses=[]; cosine=[]
    for start in range(0,len(rows),CFG['batch_size']):
        batch=rows[start:start+CFG['batch_size']]
        target=targets_for(batch,teacher)
        with torch.autocast('cuda',dtype=torch.bfloat16):
            raw=model(flatten_pairs(batch))
            loss,_=objective(model,raw,target,batch)
            aligned=F.normalize(model.align(raw).float(),dim=-1)
        z=F.normalize(raw.float(),dim=-1).cpu(); n=len(batch)
        q.append(z[:n]);p.append(z[n:2*n]);losses.append((loss.item(),n))
        cosine.append((aligned*F.normalize(target.float(),dim=-1)).sum(-1).cpu())
    q,p=torch.cat(q),torch.cat(p); sim=q@p.T
    # Multiple identical positive texts are considered relevant.
    positive_ids=[text_id(r['positive']) for r in rows]
    hit=sum(positive_ids[int(j)]==positive_ids[i] for i,j in enumerate(sim.argmax(1))) / len(rows)
    result=dict(loss=sum(l*n for l,n in losses)/sum(n for _,n in losses),positive_retrieval_hit1=hit,
                aligned_teacher_cosine=torch.cat(cosine).mean().item(),examples=len(rows))
    print(split,result,flush=True)
    return result

In [16]:
def train_stage(model, stage, steps, lr, teacher=None, dimensions=None):
    model.to(DEVICE).requires_grad_(True)
    if teacher is not None:
        assert model is not teacher
        teacher.to(DEVICE).eval().requires_grad_(False)
    optimizer=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=CFG['weight_decay'])
    scheduler=get_cosine_schedule_with_warmup(optimizer,int(steps*CFG['warmup_fraction']),steps)
    sampler=StatefulSampler(len(corpus['train']),CFG['batch_size'],CFG['seed'])
    state=load_checkpoint(stage)
    start=0;best_metric=-float('inf');best_step=0
    if state is not None:
        assert state['total_steps']==steps and state['spec']==model.spec
        model.load_state_dict(state['model'],strict=True)
        optimizer.load_state_dict(state['optimizer']);scheduler.load_state_dict(state['scheduler'])
        sampler.load_state_dict(state['sampler']);restore_rng(state['rng'])
        start=state['step'];best_metric=state['best_metric'];best_step=state['best_step'];del state
        print('Resuming',stage,start,'/',steps,flush=True)
    else:
        validate(model,teacher)
    last_step=start
    try:
        for step in range(start+1,steps+1):
            model.train();optimizer.zero_grad(set_to_none=True);running=0.0
            # Save rollback state: an interrupt in an accumulation window must not skip examples.
            boundary_sampler=copy.deepcopy(sampler.state_dict());boundary_rng=rng_state()
            for _ in range(CFG['accumulation']):
                rows=[corpus['train'][i] for i in sampler.next()]
                target=targets_for(rows,teacher)
                with torch.autocast('cuda',dtype=torch.bfloat16):
                    raw=model(flatten_pairs(rows));loss,parts=objective(model,raw,target,rows,dimensions)
                if not torch.isfinite(loss): raise FloatingPointError(f'{stage}: non-finite loss')
                (loss/CFG['accumulation']).backward();running+=loss.item()/CFG['accumulation']
            norm=nn.utils.clip_grad_norm_(model.parameters(),1.0,error_if_nonfinite=True)
            optimizer.step();scheduler.step();last_step=step
            if step%CFG['log_every']==0 or step==1:
                print(f'{stage} {step}/{steps} loss={running:.4f} '+str(parts),flush=True)
            improved=False
            if step%CFG['eval_every']==0 or step==steps:
                metrics=validate(model,teacher)
                with open(ROOT/'metrics.jsonl','a') as f:
                    f.write(json.dumps(dict(stage=stage,step=step,train_loss=running,validation=metrics))+'\n')
                if metrics['positive_retrieval_hit1']>best_metric:
                    best_metric=metrics['positive_retrieval_hit1'];best_step=step;improved=True
            if step%CFG['checkpoint_every']==0 or step==steps or improved:
                save_checkpoint(stage,step,model,optimizer,scheduler,sampler,best_metric,best_step,steps)
    except KeyboardInterrupt:
        # Discard incomplete gradients and restore sample order for the unfinished optimizer step.
        if 'step' in locals() and last_step<step:
            optimizer.zero_grad(set_to_none=True);sampler.load_state_dict(boundary_sampler);restore_rng(boundary_rng)
        save_checkpoint(stage,last_step,model,optimizer,scheduler,sampler,best_metric,best_step,steps)
        print('Checkpoint saved. Rerun this cell to resume.');raise
    model.eval()
    write_json(ROOT/'evaluations'/f'{stage}-heldout.json',validate(model,teacher,split='test'))
    return model

def restore_best(model,stage):
    state=load_checkpoint(stage)
    assert state is not None
    best=state['best_step']; del state
    parent=ROOT/'checkpoints'/stage
    choices=sorted(p for p in parent.iterdir() if p.name.startswith(f'{best:08d}-') and (p/'COMMITTED.json').exists())
    assert choices, 'Best checkpoint is missing'
    path=choices[-1];meta=json.loads((path/'COMMITTED.json').read_text())
    assert file_sha(path/'state.pt')==meta['sha256']
    state=torch.load(path/'state.pt',map_location='cpu',weights_only=False)
    model.load_state_dict(state['model'],strict=True)
    print('Selected',stage,'validation-best step',best)
    return model

In [17]:
# Export inference weights separately from optimizer state; preserve model and tokenizer metadata.
def export_model(model,label):
    final=ROOT/'exports'/label
    if final.exists():
        print('Export already exists:',final,'(use another label to export a new version)')
        return final
    temp=ROOT/'exports'/('.uploading-'+uuid.uuid4().hex);temp.mkdir(parents=True)
    model.tokenizer.save_pretrained(temp/'tokenizer')
    if model.family=='ettin':
        model.backbone.save_pretrained(temp/'backbone',safe_serialization=True)
    else:
        save_file({k:v.detach().cpu().contiguous() for k,v in model.backbone.state_dict().items()},str(temp/'backbone.safetensors'))
    write_json(temp/'encoder.json',dict(family=model.family,spec=model.spec,output_dim=model.output_dim,
               pooling='masked_mean_including_special_tokens',normalize=True,max_length=model.max_length,
               contract=contract,training_only_alignment_exported=False))
    # Preserve the source model card and available upstream notices with redistributed weights.
    source = NEO if model.family == 'neobert' else ETTIN
    source_card = hf_hub_download(source['id'], 'README.md', revision=source['revision'])
    shutil.copyfile(source_card, temp/'UPSTREAM_MODEL_CARD.md')
    write_json(temp/'NOTICE.json', dict(source=source, declared_model_license='MIT',
               supervisor=GTE, supervisor_license='Apache-2.0',
               experiment='Fine-tuned encoder; see upstream model card for attribution.',
               license_url='https://opensource.org/license/mit'))
    os.replace(temp,final)
    print('Inference export:',final)
    return final

def load_export(path):
    path=Path(path);meta=json.loads((path/'encoder.json').read_text())
    tok=AutoTokenizer.from_pretrained(path/'tokenizer')
    if meta['family']=='ettin':
        backbone=AutoModel.from_pretrained(path/'backbone',attn_implementation='sdpa')
    else:
        backbone=NeoBackbone(meta['spec'])
        backbone.load_state_dict(load_file(str(path/'backbone.safetensors')),strict=True)
    model = EmbeddingModel(backbone,tok,meta['output_dim'],meta['family'],meta['spec'])
    model.max_length = meta['max_length']
    return model.to(DEVICE).eval()

## Train the pretrained Ettin student
        The entire 32M backbone (or selectable 68M backbone) starts from its own pretrained weights.
        No teacher blocks are shape-projected into it. GTE and Ettin share the ModernBERT architecture family,
        but their widths and RoPE settings differ, so attention/hidden-state equality is not assumed.
        The training-only linear head aligns the student to GTE's 768 dimensions. Contrastive ranking and
        relational matching operate directly on the student vectors that will actually be deployed.

In [18]:
student = fresh_ettin().to(DEVICE)
print(count_parameters(student))
check_padding_invariance(student)
benchmark(student, 'ettin-mlm-before-training')

config.json:   0%|          | 0.00/1.24k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  128MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

... (3 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B /  128MB            

model.safetensors: downloading bytes:           |  0.00B            

ettin-mlm-before-training NanoMSMARCO 384 {'hit@1': 0.0, 'hit@5': 0.1, 'recall@10': 0.12, 'mrr@10': 0.0302, 'ndcg@10': 0.0513}
ettin-mlm-before-training NanoHotpotQA 384 {'hit@1': 0.1, 'hit@5': 0.2, 'recall@10': 0.13, 'mrr@10': 0.1307, 'ndcg@10': 0.0993}


{'NanoMSMARCO/384': {'hit@1': 0.0,
  'hit@5': 0.1,
  'recall@10': 0.12,
  'mrr@10': 0.03022222222222222,
  'ndcg@10': 0.05134036959702616},
 'NanoHotpotQA/384': {'hit@1': 0.1,
  'hit@5': 0.2,
  'recall@10': 0.13,
  'mrr@10': 0.13066666666666665,
  'ndcg@10': 0.09926724610573798}}

In [19]:
student = train_stage(student, 'student', CFG['student_steps'], CFG['student_lr'])
student = restore_best(student, 'student')
benchmark(student, 'student-best')

validation {'loss': 4.910220503807068, 'positive_retrieval_hit1': 0.2265625, 'aligned_teacher_cosine': -0.004119455348700285, 'examples': 128}
student 1/1000 loss=4.9185 {'contrastive': 3.85205078125, 'alignment': 0.9986017942428589, 'relational': 0.33929550647735596}
student 10/1000 loss=4.9260 {'contrastive': 3.853515625, 'alignment': 1.0079004764556885, 'relational': 0.322765976190567}
student 20/1000 loss=4.6794 {'contrastive': 3.61083984375, 'alignment': 1.0096452236175537, 'relational': 0.29456278681755066}
student 30/1000 loss=4.1815 {'contrastive': 3.140869140625, 'alignment': 0.9964897036552429, 'relational': 0.22066697478294373}
student 40/1000 loss=3.4623 {'contrastive': 2.439208984375, 'alignment': 0.9950829744338989, 'relational': 0.140022411942482}
student 50/1000 loss=2.4414 {'contrastive': 1.4425811767578125, 'alignment': 0.9870776534080505, 'relational': 0.05890408530831337}
student 60/1000 loss=2.6806 {'contrastive': 1.706634521484375, 'alignment': 0.970046877861023, 

{'NanoMSMARCO/384': {'hit@1': 0.24,
  'hit@5': 0.5,
  'recall@10': 0.6,
  'mrr@10': 0.36602380952380953,
  'ndcg@10': 0.42245711896305527},
 'NanoHotpotQA/384': {'hit@1': 0.6,
  'hit@5': 0.82,
  'recall@10': 0.58,
  'mrr@10': 0.6897142857142857,
  'ndcg@10': 0.5342930212472208}}

## Optional dimension Matryoshka experiment
    Disabled by default. Enable `run_matryoshka` in the configuration before starting a new run.
    Train nested output dimensions only after establishing the full-vector baseline. This does not train
    early exits or reduce backbone compute. Teacher alignment remains on the full vector; ranking trains each
    prefix. Compare every reported dimension rather than interpreting the averaged loss as full-vector quality.

In [20]:
DIMS = sorted(set([128, 256, student.output_dim]))
if CFG['run_matryoshka']:
    student = train_stage(student, 'matryoshka', CFG['matryoshka_steps'], CFG['student_lr']/2,
                          teacher=None, dimensions=DIMS)
    student = restore_best(student, 'matryoshka')
    benchmark(student, 'matryoshka-best', dims=DIMS)

## Export and verify reloading
    The export excludes Adam state and the training-only alignment head. To reload in another session,
    run the setup/configuration and model definition cells, then `load_export(path)`.
    NeoBERT's class is defined above in this notebook; its export is not a generic AutoModel package.

In [21]:
export_path = export_model(student, 'final')
probe = ['How do I reset my password?', 'Use the password recovery link.', 'It is raining outside.']
before = encode_texts(student, probe)
reloaded = load_export(export_path)
after = encode_texts(reloaded, probe)
torch.testing.assert_close(before, after, atol=1e-5, rtol=1e-4)
print('Export/reload parity passed. Similarities:', (after @ after.T).tolist())
del reloaded
gc.collect(); torch.cuda.empty_cache()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

README.md:   0%|          | 0.00/40.3k [00:00<?, ?B/s]

Inference export: /content/drive/MyDrive/encoder_experiments/ettin-pilot-01/exports/final


Loading weights:   0%|          | 0/62 [00:00<?, ?it/s]

Export/reload parity passed. Similarities: [[1.0, 0.8079913854598999, 0.3376524746417999], [0.8079913854598999, 1.0, 0.31070828437805176], [0.3376524746417999, 0.31070828437805176, 1.0]]


## Sources and scope
    - [NeoBERT](https://huggingface.co/chandar-lab/NeoBERT): model weights and code declared MIT by Chandar Lab.
      The notebook implementation independently expresses its published SDPA, adjacent-pair RoPE and SwiGLU equations
      and preserves checkpoint names. [Upstream architecture](https://huggingface.co/chandar-lab/NeoBERT/blob/5424c8efeea6491b151d62dee55a752165407430/model.py).
    - [Ettin](https://huggingface.co/jhu-clsp/ettin-encoder-32m): MIT; [upstream license](https://github.com/jhu-clsp/ettin-encoder-vs-decoder/blob/main/LICENSE).
    - [GTE supervisor](https://huggingface.co/Alibaba-NLP/gte-modernbert-base): Apache-2.0.
    - [Training triplets](https://huggingface.co/datasets/sentence-transformers/msmarco-msmarco-MiniLM-L6-v3).
    - [NanoBEIR](https://huggingface.co/datasets/sentence-transformers/NanoBEIR-en): pinned benchmark data.

    Source/model declarations do not substitute for the separate terms of data you add. Preserve upstream notices
    with redistributed artifacts. These notebooks do not publish anything or train on the benchmark labels.
    The benchmark here caps text at our configured length; its scores are not an exact reproduction of model-card scores.

## Quick independent sanity check
Run this final cell after any stage; it does not retrain the model, load a teacher, or use cached teacher targets. Select the in-memory student, `neo_teacher`, or `load_export(path)` on the first lines. Earlier setup and model definitions must have run.

It checks finite/nonzero vectors, repeatability, padding invariance, collapse, and 12 simple semantic rankings. Full-vector and truncated-vector results are reported separately. Prefix scores are exploratory until those prefixes have been trained. Results, including every example, are saved to Drive with a unique filename. Poor semantic scores are flagged for inspection rather than treated as a definitive benchmark failure.

In [22]:
# Quick independent sanity check -- run this cell after any training stage.
# Select neo_teacher explicitly to check NeoBERT before pruning, or use load_export(path).
SANITY_MODEL = globals().get('student', globals().get('neo_teacher'))
SANITY_LABEL = 'current-model'  # Change to e.g. 'teacher', 'pruned', 'student', 'matryoshka'.
assert SANITY_MODEL is not None, 'Create a model above, or set SANITY_MODEL = load_export(path).'

# Authored probes, separate from the training pairs and NanoBEIR. Do not train on these.
# (query, relevant passage, unrelated passage). A tiny diagnostic, not a quality benchmark.
SANITY_CASES = [
    ('I forgot my account password.', 'Use the password reset link to regain access to your account.',
     'The tomato plants need watering every morning.'),
    ('How can I get my money back for this purchase?', 'Request a refund for your order through customer support.',
     'The train leaves the station at six in the evening.'),
    ('My phone battery runs out too quickly.', 'Reduce screen brightness and background apps to extend battery life.',
     'The museum has opened a new exhibition of ancient pottery.'),
    ('How do I bake a loaf of bread?', 'Mix flour, water and yeast, let the dough rise, then bake it in the oven.',
     'The football team won the championship yesterday.'),
    ('I want to cancel my monthly subscription.', 'Open your billing settings and choose cancel subscription.',
     'Migrating birds travel south during the winter.'),
    ('My laptop cannot connect to Wi-Fi.', 'Check the wireless network settings and restart your router.',
     'The orchestra performed a symphony at the concert hall.'),
    ('How can I track the package I ordered?', 'Enter your parcel tracking number to see its delivery status.',
     'Stretching your legs before a run can help improve flexibility.'),
    ('How do I change the language in the app?', 'Choose your preferred display language in the application settings.',
     'The recipe calls for two eggs and a cup of milk.'),
    ('I need to book a visit with a doctor.', 'Contact the clinic to schedule a medical appointment.',
     'The new telescope can observe distant galaxies.'),
    ('Where can I find books to borrow?', 'Visit the public library to check out books with your library card.',
     'The car needs new tires before the road trip.'),
    ('How can I recover a file that I deleted?', 'Look in the recycle bin or restore the document from a backup.',
     'The beach is covered in shells after the tide goes out.'),
    ('I want to learn to play the guitar.', 'Practice basic chords and strumming patterns on your guitar each day.',
     'The city council approved construction of a new bridge.'),
]

def sanity_scores(raw, cases, dimensions):
    # Pure scoring function: no model, tokenizer or teacher dependency.
    raw = raw.detach().float().cpu()
    n = len(cases)
    assert raw.ndim == 2 and len(raw) == 3*n
    finite = bool(torch.isfinite(raw).all())
    norms = raw.norm(dim=-1)
    nonzero = finite and bool((norms > 1e-8).all())
    result = dict(finite=finite, nonzero=nonzero,
                  raw_norm_min=float(norms.min()) if finite else None,
                  raw_norm_mean=float(norms.mean()) if finite else None,
                  profiles={})
    if not nonzero:
        return result
    for dim in dimensions:
        prefix_norms = raw[:, :dim].norm(dim=-1)
        z = F.normalize(raw[:, :dim], dim=-1)
        q, positive, unrelated = z.split(n)
        pos = (q*positive).sum(-1); neg = (q*unrelated).sum(-1)
        scores = q @ torch.cat((positive, unrelated)).T
        positive_mask = torch.zeros_like(scores, dtype=torch.bool)
        positive_mask[torch.arange(n), torch.arange(n)] = True
        # Conservative tie handling: a collapsed model must not pass by index order.
        ranks = 1 + ((scores >= pos[:,None]-1e-6) & ~positive_mask).sum(-1)
        pairwise = z @ z.T
        offdiag = pairwise[~torch.eye(len(z),dtype=torch.bool)]
        result['profiles'][str(dim)] = dict(
            nonzero_prefix=bool((prefix_norms > 1e-8).all()),
            easy_accuracy=float((pos > neg+1e-6).float().mean()),
            mean_margin=float((pos-neg).mean()),
            hit_at_1=float((ranks <= 1).float().mean()),
            hit_at_5=float((ranks <= 5).float().mean()),
            mrr=float((1/ranks.float()).mean()),
            normalized_spread=float((z-z.mean(0)).square().sum(-1).mean()),
            off_diagonal_cosine_mean=float(offdiag.mean()),
            examples=[dict(query=cases[i][0], positive=cases[i][1], unrelated=cases[i][2],
                           positive_cosine=float(pos[i]), unrelated_cosine=float(neg[i]),
                           positive_rank=int(ranks[i]), easy_correct=bool(pos[i]>neg[i]+1e-6))
                      for i in range(n)])
    return result

@torch.inference_mode()
def run_bottom_sanity(model, label):
    was_training = model.training
    model.eval()
    model_device = next(model.parameters()).device
    def encode_raw(texts):
        outputs = []
        for start in range(0,len(texts),4):
            ctx = torch.autocast('cuda',dtype=torch.bfloat16) if model_device.type == 'cuda' else nullcontext()
            with ctx:
                outputs.append(model(texts[start:start+4]).float().cpu())
        return torch.cat(outputs)
    try:
        # Same order as the scoring function: queries, positives, unrelated passages.
        texts = [row[column] for column in range(3) for row in SANITY_CASES]
        raw = encode_raw(texts)
        dimensions = sorted(set([d for d in (128,256,384) if d < raw.shape[1]] + [raw.shape[1]]))
        report = sanity_scores(raw,SANITY_CASES,dimensions)
        report.update(label=label, family=getattr(model,'family','unknown'),
                      contract=globals().get('CONTRACT_ID'), cases=len(SANITY_CASES),
                      candidate_passages=2*len(SANITY_CASES),
                      note='Small authored diagnostic. Truncated profiles are exploratory unless trained for them.')
        short = SANITY_CASES[0][0]
        long = 'This is a much longer passage about a library, its collection of books, the weather outside, and plans for a journey. '
        a = encode_raw([short]); repeated = encode_raw([short]); padded = encode_raw([short,long])[:1]
        repeat_ok = bool(torch.isfinite(a).all() and torch.isfinite(repeated).all() and
                         torch.allclose(a,repeated,atol=1e-5,rtol=1e-4))
        padding_cosine = float(F.cosine_similarity(a,padded).item()) if bool(torch.isfinite(a).all() and torch.isfinite(padded).all()) else None
        report['repeatable'] = repeat_ok
        report['padding_cosine'] = padding_cosine
        report['plumbing_passed'] = bool(report['finite'] and report['nonzero'] and repeat_ok and
                                         padding_cosine is not None and padding_cosine > .999)
        flags = []
        for dim, metrics in report['profiles'].items():
            if not metrics['nonzero_prefix'] or metrics['normalized_spread'] < 1e-4:
                flags.append(f'{dim}D: zero or nearly collapsed embeddings.')
            if metrics['easy_accuracy'] <= .5:
                flags.append(f'{dim}D: easy semantic ranking at or below 50%; inspect examples.')
        report['flags'] = flags
        path = ROOT/'evaluations'/f'sanity-{time.time_ns()}.json'
        write_json(path,report)
        print('Plumbing:', 'PASS' if report['plumbing_passed'] else 'FAIL',
              '| repeatable:', repeat_ok, '| single/padded cosine:', padding_cosine)
        print(' dim   easy    hit@1   hit@5    margin    spread')
        for dim,metrics in report['profiles'].items():
            print(f"{dim:>4}  {metrics['easy_accuracy']:6.1%}  {metrics['hit_at_1']:6.1%}  "
                  f"{metrics['hit_at_5']:6.1%}  {metrics['mean_margin']:8.4f}  {metrics['normalized_spread']:8.4f}")
        if report['profiles']:
            print('\nFull-vector examples (positive should outrank unrelated):')
            for row in report['profiles'][str(raw.shape[1])]['examples']:
                print(f"{'OK' if row['easy_correct'] else 'MISS':4}  positive={row['positive_cosine']:.3f} "
                      f"unrelated={row['unrelated_cosine']:.3f} rank={row['positive_rank']:2}  {row['query']}")
        for flag in flags: print('Inspect:',flag)
        print('\nSaved to Drive:',path)
        print('Passing the plumbing checks does not establish semantic quality; use the retrieval benchmarks too.')
        if not report['plumbing_passed']:
            raise RuntimeError(f'Numerical, repeatability or padding check failed. Details: {path}')
        return report
    finally:
        model.train(was_training)

sanity_report = run_bottom_sanity(SANITY_MODEL,SANITY_LABEL)


Plumbing: PASS | repeatable: True | single/padded cosine: 0.9999837279319763
 dim   easy    hit@1   hit@5    margin    spread
 128  100.0%   91.7%  100.0%    0.3852    0.5493
 256  100.0%   91.7%  100.0%    0.3824    0.5251
 384  100.0%   91.7%  100.0%    0.3599    0.5016

Full-vector examples (positive should outrank unrelated):
OK    positive=0.857 unrelated=0.370 rank= 1  I forgot my account password.
OK    positive=0.752 unrelated=0.434 rank= 1  How can I get my money back for this purchase?
OK    positive=0.840 unrelated=0.443 rank= 1  My phone battery runs out too quickly.
OK    positive=0.786 unrelated=0.367 rank= 1  How do I bake a loaf of bread?
OK    positive=0.841 unrelated=0.413 rank= 1  I want to cancel my monthly subscription.
OK    positive=0.787 unrelated=0.391 rank= 1  My laptop cannot connect to Wi-Fi.
OK    positive=0.687 unrelated=0.426 rank= 1  How can I track the package I ordered?
OK    positive=0.764 unrelated=0.333 rank= 1  How do I change the language in the a